In [33]:
from dotenv import load_dotenv
load_dotenv()
from ingest import load_faq_data, build_index
from rag_helper import RagHelper
from openai import OpenAI
import os


In [34]:

documents = load_faq_data()
index = build_index(documents)



In [35]:

openai_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.
Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""


In [40]:
assistant = RagHelper(
    index=index,
    llm_client=openai_client,
    instructions=INSTRUCTIONS,
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
)

In [42]:
answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [60]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [58]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}

In [62]:
response = openai_client.chat.completions.create(
        model="nvidia/nemotron-3-ultra-550b-a55b:free",
        messages=[{"role": "user", "content": "I just discovered the course. Can I join now?"}],
        tools=[search_tool],
    )
call=response.choices[0].message.tool_calls
print(call)


[ChatCompletionMessageFunctionToolCall(id='call-41a77f1c-6c18-415f-bbe0-454dddf44227', function=Function(arguments='{"query":"join course now late enrollment"}', name='search'), type='function', index=0)]


In [ ]:
import json

# Step 1: get the first tool call (there could be more than one, but let's handle one for now)
tool_call = response.choices[0].message.tool_calls[0]

# Step 2: parse the arguments the model suggested
args = json.loads(tool_call.function.arguments)

# Step 3: actually run your real search function with those arguments
results = search(**args)
result_json = json.dumps(results, indent=2)



In [ ]:
# Step 4: build the message history to send back —
# must include the original assistant message that contained the tool call,
# followed by a "tool" role message with the result

messages = [
    {"role": "user", "content": "I just discovered the course. Can I join now?"},
    response.choices[0].message,  # the assistant's tool-call message
    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result_json,
    },
]

# Step 5: call the model again with the results included — now it can give a real answer
final_response = openai_client.chat.completions.create(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    messages=messages,
    tools=[search_tool],
)

print(final_response.choices[0].message.content)

Yes, you can absolutely join the LLM Zoomcamp now! The course materials are publicly available and you can start learning at any time.

**Key points:**
- **Yes, you can join** – the course is self-paced and materials are accessible whenever you're ready
- **Certificate requirement** – if you want to earn a certificate, you'll need to submit your capstone project while a live cohort is still accepting submissions (check the [course platform](https://courses.datatalks.club/llm-zoomcamp-2026/) for current deadlines)
- **No registration needed** – you don't need to wait for a confirmation email; you can just start learning and submitting homework while the forms are open

**To get started:**
1. Visit the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/)
2. Check the [general Zoomcamp logistics](https://datatalks.club/docs/courses/zoomcamp-logistics/)
3. Explore the [GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp)

**Typical workflow:**
1. Watch lesson

In [66]:
messages 

[{'role': 'user', 'content': 'I just discovered the course. Can I join now?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call-41a77f1c-6c18-415f-bbe0-454dddf44227', function=Function(arguments='{"query":"join course now late enrollment"}', name='search'), type='function', index=0)], reasoning='The user is asking if they can join the course now. I need to search the FAQ database for information about joining the course, late enrollment, or similar topics. Let me search for relevant FAQ entries.', reasoning_details=[{'type': 'reasoning.text', 'text': 'The user is asking if they can join the course now. I need to search the FAQ database for information about joining the course, late enrollment, or similar topics. Let me search for relevant FAQ entries.', 'format': 'unknown', 'index': 0}]),
 {'role': 'tool',
  'tool_call_id': 'call-41a77f1c-6c18-415f-bbe0-454dd

## agentic loop

In [80]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [86]:
question =  "How do I run Olama locally?"

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question},
]

In [87]:
response = openai_client.chat.completions.create(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    messages=messages,
    tools=[search_tool],
)

In [83]:
print(response.choices[0].message.content)

None


In [84]:
def make_call(tool_call):
    args = json.loads(tool_call.function.arguments)

    if tool_call.function.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result_json,
    }

In [88]:
while True:
    response = openai_client.chat.completions.create(
        model="nvidia/nemotron-3-ultra-550b-a55b:free",
        messages=messages,
        tools=[search_tool],
    )

    message = response.choices[0].message
    messages.append(message)

    if message.tool_calls:
        for tool_call in message.tool_calls:
            print("function_call:", tool_call.function.name, tool_call.function.arguments)
            call_output = make_call(tool_call)
            messages.append(call_output)
    else:
        print("ASSISTANT:")
        print(message.content)
        break

function_call: search {"query":"Ollama run locally"}
ASSISTANT:
Based on the course FAQ, here's how to run **Ollama** locally:

## Installation

**Visit [ollama.com/download](https://ollama.com/download)** and choose your OS:

- **macOS**: Download the `.pkg` and install it
- **Windows**: Download the `.msi` and install it  
- **Linux**: Run in terminal:
  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```

## Running a Model

After installation, open a terminal and run:

```bash
ollama run llama3
```

This will:
- Download the LLaMA 3 model (~4GB)
- Start the model locally
- Open a chat interface where you can type questions

## Test the Local Server

```bash
curl http://localhost:11434
```
You should get a JSON response showing available models.

## Python Client

Install the Python library:
```bash
pip install ollama
```

Then use it in your code:
```python
import ollama

response = ollama.chat(
    model='llama3',
    messages=[{"role": "user", "content": "your prompt he

## toyaikit

In [89]:
%pip install toyaikit

   ---------------------------------------- 0.0/998.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/998.3 kB ? eta -:--:--
   --------------------- ------------------ 524.3/998.3 kB 3.3 MB/s eta 0:00:01
   ------------------------------- -------- 786.4/998.3 kB 2.3 MB/s eta 0:00:01
   ---------------------------------------- 998.3/998.3 kB 2.5 MB/s  0:00:00

   ----- ---------------------------------- 1/7 [docstring-parser]
   ----------- ---------------------------- 2/7 [httpcore2]
   ----------- ---------------------------- 2/7 [httpcore2]
   ----------- ---------------------------- 2/7 [httpcore2]
   ----------------- ---------------------- 3/7 [httpx2]
   ----------------- ---------------------- 3/7 [httpx2]
   ----------------- ---------------------- 3/7 [httpx2]
   ---------------------- ----------------- 4/7 [anthropic]
   ---------------------- ----------------- 4/7 [anthropic]
   ---------------------- ----------------- 4/7 [anthropic]
   -------------------

In [90]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [91]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [92]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [93]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [94]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [97]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

from toyaikit.llm import OpenAIClient

llm_client = OpenAIClient(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",  # use your working OpenRouter model, not gpt-5.4-mini
    client=openai_client  # reuse the client you already configured for OpenRouter
)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [98]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


f:\VS code files - TRYING\llm-zoomcamp-2026-code\.venv\Lib\site-packages\toyaikit\chat\runners.py:283: UnknownModelWarning: No pricing data for model 'free'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [99]:
runner.run();

-> Response received


f:\VS code files - TRYING\llm-zoomcamp-2026-code\.venv\Lib\site-packages\toyaikit\chat\runners.py:283: UnknownModelWarning: No pricing data for model 'free'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


-> Response received


-> Response received


-> Response received


...

-> Response received


-> Response received


f:\VS code files - TRYING\llm-zoomcamp-2026-code\.venv\Lib\site-packages\toyaikit\chat\runners.py:283: UnknownModelWarning: No pricing data for model 'free'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


-> Response received


KeyboardInterrupt: Interrupted by user